In [ ]:
!pip install transformers==4.30.2

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
import numpy as np
import random
from tqdm.auto import tqdm
import json
from typing import List, Tuple, Dict
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
"""
Fine-tune T5 for Topic Labeling from Topic Words
Optimized for small datasets (200 samples)
UPDATED: Fixes catastrophic forgetting and word repetition issues
"""
class TopicLabelingDataset(Dataset):
    """Dataset for topic word to label mapping"""
    
    def __init__(
        self, 
        topic_words: List[List[str]], 
        topic_labels: List[str],
        tokenizer: T5Tokenizer,
        max_source_length: int = 128,
        max_target_length: int = 32,
        augment: bool = False
    ):
        self.topic_words = topic_words
        self.topic_labels = topic_labels
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length
        self.augment = augment
        
    def __len__(self):
        return len(self.topic_words)
    
    def augment_topic_words(self, words: List[str]) -> List[str]:
        """Apply data augmentation to topic words"""
        words = words.copy()
        
        # 50% chance to shuffle word order
        if random.random() < 0.5:
            random.shuffle(words)
        
        # 30% chance to drop one word (if we have more than 3 words)
        if len(words) > 3 and random.random() < 0.3:
            words = words[:-1]
        
        # 20% chance to add synonyms or related terms (if available)
        # This requires a synonym dictionary - placeholder for now
        
        return words
    
    def add_semantic_context(self, words: List[str], label: str = None) -> str:
        """Add rich semantic context to help model understand the task"""
        # Create a more descriptive prompt that guides the model
        context_templates = [
            f"Generate a concise topic label that describes the main theme of these keywords: {', '.join(words)}",
            f"What is the overarching topic for these terms: {', '.join(words)}? Provide a brief label.",
            f"Summarize the common theme of: {', '.join(words)} in 2-4 words.",
            f"Topic keywords: {', '.join(words)}. Generate a descriptive label for this topic.",
        ]
        
        # Use different templates during training for variety
        return random.choice(context_templates) if label else context_templates[0]
    
    def __getitem__(self, idx):
        words = self.topic_words[idx]
        label = self.topic_labels[idx]
        
        # Apply augmentation during training
        if self.augment:
            words = self.augment_topic_words(words)
        
        # Use rich semantic prompt instead of minimal format
        source_text = f"Generate a concise topic label that describes the main theme of these keywords: {', '.join(words)}"
        
        # Tokenize input
        source = self.tokenizer(
            source_text,
            max_length=self.max_source_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize output
        target = self.tokenizer(
            label,
            max_length=self.max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Replace padding token id's of the labels by -100
        # so they are ignored by the loss function
        labels = target['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': source['input_ids'].squeeze(),
            'attention_mask': source['attention_mask'].squeeze(),
            'labels': labels
        }

In [ ]:
class T5TopicLabeler:
    """Fine-tune T5 for topic labeling"""
    
    def __init__(
        self,
        model_name: str = 't5-small',
        learning_rate: float = 5e-5,  # MUCH lower to preserve pre-trained knowledge
        weight_decay: float = 0.01,
        max_source_length: int = 128,
        max_target_length: int = 32,
        device: str = None,
        freeze_encoder: bool = False,  # Option to freeze encoder layers
        freeze_layers: int = 0  # Number of encoder layers to freeze
    ):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Using device: {self.device}")
        
        # Load model and tokenizer
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        
        # Freeze encoder to preserve semantic understanding
        if freeze_encoder:
            logger.info("Freezing entire encoder")
            for param in self.model.encoder.parameters():
                param.requires_grad = False
        elif freeze_layers > 0:
            logger.info(f"Freezing first {freeze_layers} encoder layers")
            for i in range(freeze_layers):
                for param in self.model.encoder.block[i].parameters():
                    param.requires_grad = False
        
        self.model.to(self.device)
        
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length
        
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        
    def prepare_data(
        self,
        topic_words: List[List[str]],
        topic_labels: List[str],
        test_size: float = 0.15,
        batch_size: int = 8
    ) -> Tuple[DataLoader, DataLoader]:
        """Prepare train and validation dataloaders"""
        
        # Convert to list if pandas Series/DataFrame (fixes indexing issue)
        if hasattr(topic_words, 'tolist'):
            topic_words = topic_words.tolist()
        if hasattr(topic_labels, 'tolist'):
            topic_labels = topic_labels.tolist()
        
        # Split data
        train_words, val_words, train_labels, val_labels = train_test_split(
            topic_words, topic_labels, 
            test_size=test_size, 
            random_state=42
        )
        
        # Convert to lists (important for proper indexing in Dataset)
        train_words = list(train_words)
        val_words = list(val_words)
        train_labels = list(train_labels)
        val_labels = list(val_labels)
        
        logger.info(f"Train samples: {len(train_words)}, Validation samples: {len(val_words)}")
        
        # Create datasets with augmentation for training
        train_dataset = TopicLabelingDataset(
            train_words, train_labels, 
            self.tokenizer,
            self.max_source_length,
            self.max_target_length,
            augment=True  # Enable augmentation for training
        )
        
        val_dataset = TopicLabelingDataset(
            val_words, val_labels,
            self.tokenizer,
            self.max_source_length,
            self.max_target_length,
            augment=False  # No augmentation for validation
        )
        
        # Create dataloaders
        train_loader = DataLoader(
            train_dataset, 
            batch_size=batch_size, 
            shuffle=True
        )
        
        val_loader = DataLoader(
            val_dataset, 
            batch_size=batch_size, 
            shuffle=False
        )
        
        return train_loader, val_loader
    
    def train_epoch(self, train_loader, optimizer, scheduler):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc="Training")
        
        for batch in progress_bar:
            # Move batch to device
            batch = {k: v.to(self.device) for k, v in batch.items()}
            
            # Forward pass
            outputs = self.model(**batch)
            loss = outputs.loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        return total_loss / len(train_loader)
    
    def validate(self, val_loader):
        """Validate the model"""
        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                total_loss += outputs.loss.item()
        
        return total_loss / len(val_loader)
    
    def train(
        self,
        topic_words: List[List[str]],
        topic_labels: List[str],
        num_epochs: int = 15,  # Reduced epochs to prevent overwriting
        batch_size: int = 8,
        patience: int = 7,  # Increased patience
        save_path: str = 'best_model',
        gradient_accumulation_steps: int = 1
    ):
        """Complete training loop with early stopping"""
        
        # Prepare data
        train_loader, val_loader = self.prepare_data(
            topic_words, topic_labels, 
            batch_size=batch_size
        )
        
        # Setup optimizer - only optimize unfrozen parameters
        optimizer = AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.learning_rate,
            weight_decay=self.weight_decay
        )
        
        # Setup learning rate scheduler
        total_steps = len(train_loader) * num_epochs // gradient_accumulation_steps
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=total_steps // 10,  # 10% warmup
            num_training_steps=total_steps
        )
        
        # Training loop
        history = {'train_loss': [], 'val_loss': []}
        
        for epoch in range(num_epochs):
            logger.info(f"\nEpoch {epoch + 1}/{num_epochs}")
            
            # Train
            train_loss = self.train_epoch(train_loader, optimizer, scheduler)
            history['train_loss'].append(train_loss)
            
            # Validate
            val_loss = self.validate(val_loader)
            history['val_loss'].append(val_loss)
            
            logger.info(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            
            # Early stopping check
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                
                # Save best model
                self.save_model(save_path)
                logger.info(f"✓ New best model saved (val_loss: {val_loss:.4f})")
            else:
                self.patience_counter += 1
                logger.info(f"No improvement. Patience: {self.patience_counter}/{patience}")
                
                if self.patience_counter >= patience:
                    logger.info(f"Early stopping triggered after {epoch + 1} epochs")
                    break
        
        # Save training history
        with open(f"{save_path}/training_history.json", 'w') as f:
            json.dump(history, f, indent=2)
        
        logger.info(f"\nTraining completed! Best validation loss: {self.best_val_loss:.4f}")
        return history
    
    def predict(
        self, 
        topic_words: List[str],
        num_beams: int = 5,
        num_return_sequences: int = 1,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 0.95,
        repetition_penalty: float = 2.0,  # Penalize repetition
        length_penalty: float = 1.0,
        no_repeat_ngram_size: int = 2  # Prevent repeating phrases
    ) -> str:
        """Generate topic label for given topic words"""
        self.model.eval()
        
        # Use rich semantic prompt
        source_text = f"Generate a concise topic label that describes the main theme of these keywords: {', '.join(topic_words)}"
        
        # Tokenize
        inputs = self.tokenizer(
            source_text,
            max_length=self.max_source_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(self.device)
        
        # Generate with constraints to avoid copying input
        with torch.no_grad():
            outputs = self.model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=self.max_target_length,
                min_length=2,  # Ensure at least 2 tokens
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                repetition_penalty=repetition_penalty,  # Strongly discourage repetition
                length_penalty=length_penalty,
                no_repeat_ngram_size=no_repeat_ngram_size,  # No repeated bigrams
                early_stopping=True,
                do_sample=False  # Use greedy/beam search for consistency
            )
        
        # Decode
        label = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return label
    
    def batch_predict(
        self,
        topic_words_list: List[List[str]],
        batch_size: int = 16
    ) -> List[str]:
        """Generate labels for multiple topic word sets"""
        predictions = []
        
        for i in tqdm(range(0, len(topic_words_list), batch_size), desc="Predicting"):
            batch = topic_words_list[i:i+batch_size]
            for words in batch:
                pred = self.predict(words)
                predictions.append(pred)
        
        return predictions
    
    def save_model(self, save_path: str):
        """Save model and tokenizer"""
        self.model.save_pretrained(save_path)
        self.tokenizer.save_pretrained(save_path)
    
    def load_model(self, load_path: str):
        """Load saved model and tokenizer"""
        self.model = T5ForConditionalGeneration.from_pretrained(load_path)
        self.tokenizer = T5Tokenizer.from_pretrained(load_path)
        self.model.to(self.device)
        logger.info(f"Model loaded from {load_path}")

In [ ]:
verified_df = pd.read_csv("/kaggle/input/scientific-topic-labeling-dataset/verified.csv")
verified_df.head()

In [ ]:
taxonomy_df = pd.read_csv("/kaggle/input/scientific-topic-labeling-dataset/taxonomy_normalized.csv")
taxonomy_df.head()

In [ ]:
taxonomy_df = taxonomy_df.dropna(subset=['topic_words', 'topic_label'])
print(len(taxonomy_df))

In [ ]:
taxonomy_tw = taxonomy_df['topic_words'].tolist()
taxonomy_tw = [item.split(',') for item in taxonomy_tw]
taxonomy_tw[0:2]

In [ ]:
verified_tw = verified_df['topic_words'].tolist()
verified_tw = [item.split(',') for item in verified_tw]
verified_tw[0:2]

In [ ]:
taxonomy_tl = taxonomy_df['topic_label'].tolist()
taxonomy_tl[0:2]

In [ ]:
verified_tl = verified_df['topic_label'].tolist()
verified_tl[0:2]

### T5-large

In [ ]:
model_name='t5-large'

##### Taxonomy

In [ ]:
taxonomy_trainer_large = T5TopicLabeler(
    model_name=model_name,
    learning_rate=5e-5,
    weight_decay=0.01,
    freeze_layers=4  # freeze first 4 encoder layers to preserve semantics
)

In [ ]:
taxonomy_history_large = taxonomy_trainer_large.train(
    topic_words=taxonomy_tw,
    topic_labels=taxonomy_tl,
    num_epochs=15,
    batch_size=1,
    patience=7,
    save_path='taxonomy_large_model'
)